# Try FitRank in 2 minutes

[![PyPI](https://img.shields.io/pypi/v/fitrank)](https://pypi.org/project/fitrank/)
[![Docs](https://img.shields.io/badge/docs-EN%20%7C%20FR-blue)](https://adeshinan.github.io/fitrank/)
[![GitHub](https://img.shields.io/github/stars/Adeshinan/fitrank?style=social)](https://github.com/Adeshinan/fitrank)

**FitRank** trains, compares and ranks scikit-learn models from a single config: same split, same folds and same metric for every model, and no hidden preprocessing.

In this notebook you will:
1. rank 5 models on a clean dataset (Iris);
2. see how FitRank handles real-world data with missing values and text (Titanic);
3. run the same experiment from a YAML file and the command line;
4. predict new rows with the saved best model;
5. add your own model in 5 lines.

👉 **Runtime → Run all**, then scroll down. Nothing to install on your machine.

In [ ]:
%pip install -q fitrank
import fitrank

print("FitRank", fitrank.__version__)

## 1. Rank models on Iris

Give FitRank a DataFrame, the column to predict and the models to compare. It keeps 20 % of the rows aside for a final test score, tunes each model with 5-fold cross-validation, and ranks them by weighted F1 score.

In [ ]:
from sklearn.datasets import load_iris

from fitrank import Benchmark

iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={"target": "species"})
df["species"] = df["species"].map(dict(enumerate(iris.target_names)))

bench = Benchmark(
    data=df,
    target="species",
    task="classification",
    models=["logistic_regression", "svm", "knn", "decision_tree", "random_forest"],
)
results = bench.run(export=False)

# Main columns; results.leaderboard() has them all (precision, recall, errors...)
cols = ["rank", "model", "cv_f1_weighted", "cv_std", "test_f1_weighted", "fit_s", "best_params"]
results.leaderboard()[cols].round(3)

The best model is a regular scikit-learn `Pipeline`, already trained and ready to use:

In [ ]:
best = results.best_model()
best

## 2. Real-world data: missing values and text

The Titanic dataset has missing ages and text columns (`sex`, `embarked`). Many tools would fix these silently. **FitRank never changes your data unless you say so**: it stops and tells you exactly what to declare.

In [ ]:
import seaborn as sns

columns = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
titanic = sns.load_dataset("titanic")[columns]
print(titanic.isna().sum())
titanic.head()

In [ ]:
from fitrank import DataError

try:
    Benchmark(
        data=titanic,
        target="survived",
        task="classification",
        models=["logistic_regression", "random_forest"],
    ).run(export=False)
except DataError as error:
    print(error)

Let's declare the preprocessing explicitly: fill and scale the numbers, fill and encode the text. Each step is fitted inside the cross-validation folds, so nothing leaks from the test data.

In [ ]:
preprocessing = {
    "numeric": {
        "columns": ["pclass", "age", "sibsp", "parch", "fare"],
        "steps": [{"imputer": {"strategy": "median"}}, "scaler"],
    },
    "categorical": {
        "columns": ["sex", "embarked"],
        "steps": [{"imputer": {"strategy": "most_frequent"}}, "encoder"],
    },
}

results = Benchmark(
    data=titanic,
    target="survived",
    task="classification",
    models=["logistic_regression", "svm", "knn", "random_forest", "gradient_boosting"],
    preprocessing=preprocessing,
).run(export=False)
results.leaderboard()[cols].round(3)

## 3. The same experiment in a YAML file

A YAML config is a recipe you can version, share and rerun: same config, same ranking. The command line prints the leaderboard and saves everything (config, results, best model) in a `runs/` folder.

In [ ]:
titanic.to_csv("titanic.csv", index=False)

In [ ]:
%%writefile titanic.yaml
project:
  name: titanic

data:
  path: titanic.csv
  target: survived

task: classification

preprocessing:
  numeric:
    columns: [pclass, age, sibsp, parch, fare]
    steps: [{imputer: {strategy: median}}, scaler]
  categorical:
    columns: [sex, embarked]
    steps: [{imputer: {strategy: most_frequent}}, encoder]

models:
  - name: logistic_regression
  - name: svm
  - name: random_forest
    params: {n_estimators: [100, 300], max_depth: [null, 5]}
  - name: gradient_boosting

In [ ]:
!fitrank run titanic.yaml

## 4. Predict new rows with the saved model

The run folder contains `best_model.joblib`. Load it and predict, here on five passengers (without the `survived` column).

In [ ]:
from pathlib import Path

from fitrank import load_model

model_path = sorted(Path("runs").glob("titanic_*/best_model.joblib"))[-1]
model = load_model(model_path)

passengers = titanic.drop(columns="survived").head(5)
passengers.assign(predicted_survived=model.predict(passengers))

## 5. Add your own model

Any scikit-learn estimator can join the comparison: register it with its hyperparameter grid, then use its name like a built-in model.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

from fitrank import register_model


@register_model("extra_trees", task="classification")
def extra_trees():
    return ExtraTreesClassifier(random_state=0), {
        "n_estimators": [100, 300],
        "max_depth": [None, 5],
    }


results = Benchmark(
    data=titanic,
    target="survived",
    task="classification",
    models=["random_forest", "gradient_boosting", "extra_trees"],
    preprocessing=preprocessing,
).run(export=False)
results.leaderboard()[cols].round(3)

## Next steps

- 📖 **Documentation** (English and French): https://adeshinan.github.io/fitrank/
- ⭐ **Star the project** if it helped you: https://github.com/Adeshinan/fitrank
- 💬 **Feedback, bugs, ideas**: https://github.com/Adeshinan/fitrank/issues
- 🖥️ **On your machine**: `pip install fitrank`, then `fitrank init --data your_data.csv --target your_column` writes a commented starter config.

FitRank is open source (MIT).